# 7b. Single-Cell QC

## Purpose
Flag low-quality single cells per patient using three criteria applied in cascade:
1. **NaN detection** — cells missing key metadata or feature values
2. **Inherited organoid flags** — cells whose parent organoid was flagged in `7a`
3. **Nucleus outliers** — abnormally small/large nuclei or high mass displacement

Outlier detection (step 3) only runs on cells that passed steps 1 and 2.

This is **step 7b of Stage 4 (image-based profiling)**. It runs once per patient
and depends on `7a.organoid_qc.ipynb` having run first.

## Inputs
- `data/{patient}/image_based_profiles/3.annotated_profiles/sc_anno.parquet`
- `data/{patient}/image_based_profiles/4.qc_profiles/organoid_flagged_outliers.parquet`

## Outputs
- `data/{patient}/image_based_profiles/4.qc_profiles/sc_flagged_outliers.parquet`
  — SC profile with added `Metadata_cqc_*` flag columns

## Notes
- QC flags are additive: a cell can be flagged by multiple criteria simultaneously.
- The `Metadata_cqc_organoid_flagged` column propagates organoid-level flags down
  to all cells belonging to that organoid, linking 7a and 7b outputs.

In [1]:
import os
import pathlib

import pandas as pd
from cosmicqc import find_outliers
from image_analysis_3D.file_utils.arg_parsing_utils import parse_args
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()

profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot/NF1_organoid_data")).resolve(),
    root_dir,
)
# profile_base_dir = root_dir
print(profile_base_dir)

/home/jenna/mnt/bandicoot/NF1_organoid_data


In [2]:
if not in_notebook:
    args = parse_args()
    patient = args["patient"]
    image_based_profiles_subparent_name = args["image_based_profiles_subparent_name"]

else:
    patient = "SARCO361_T1"
    image_based_profiles_subparent_name = "image_based_profiles"

In [3]:
import json

# Per-patient single-cell outlier z-score thresholds. Patients are tuned individually
# by visually inspecting flagged nuclei and adjusting their entry in this file.
sc_outlier_thresholds_path = (
    root_dir
    / "4.processing_image_based_profiles"
    / "data"
    / "qc_thresholds"
    / "single_cell_outlier_thresholds.json"
).resolve(strict=True)
with open(sc_outlier_thresholds_path) as f:
    sc_outlier_thresholds = json.load(f)

if patient not in sc_outlier_thresholds:
    raise ValueError(
        f"No single-cell outlier thresholds configured for patient '{patient}' in "
        f"{sc_outlier_thresholds_path}. Add an entry for this patient before running QC."
    )

patient_sc_thresholds = sc_outlier_thresholds[patient]
small_nuclei_threshold = patient_sc_thresholds["small_nuclei_volume"]
large_nuclei_threshold = patient_sc_thresholds["large_nuclei_volume"]
high_mass_displacement_threshold = patient_sc_thresholds["high_mass_displacement"]
print(f"Using single-cell outlier thresholds for {patient}: {patient_sc_thresholds}")

Using single-cell outlier thresholds for SARCO361_T1: {'small_nuclei_volume': -0.5, 'large_nuclei_volume': 2, 'high_mass_displacement': 2}


## Load profiles and initialize QC flags

QC is applied in three rounds:
1. **NaN detection** (`Metadata_cqc_nan_detected`) — missing ObjectID, volume, or parent
2. **Inherited organoid flags** (`Metadata_cqc_organoid_flagged`, `Metadata_cqc_missing_parent_organoid`)
   — cells whose parent organoid failed QC in 7a, or have no parent organoid at all
3. **Nucleus outliers** — applied only to cells that passed rounds 1 and 2

In [4]:
sc_file = pathlib.Path(
    profile_base_dir
    / "data"
    / "image_based_profiles_production_zedprofiler_pipeline_20260915_f5a6f16"
    / f"{patient}"
    # / f"{image_based_profiles_subparent_name}"
    / "3.annotated_profiles/sc_anno.parquet"
)
organoid_file = pathlib.Path(
    profile_base_dir
    / "data"
    / "image_based_profiles_production_zedprofiler_pipeline_20260915_f5a6f16"
    / f"{patient}"
    # / f"{image_based_profiles_subparent_name}"
    / "4.qc_profiles/organoid_flagged_outliers.parquet"
)

nucleocentric_annotated_sammed_path = pathlib.Path(
    profile_base_dir
    / "data"
    / "image_based_profiles_production_zedprofiler_pipeline_20260915_f5a6f16"
    / f"{patient}"
    # / f"{image_based_profiles_subparent_name}"
    / "3.annotated_profiles/nucleocentric_sammed_anno.parquet"
).resolve()
nucleocentric_annotated_morphem_output_path = pathlib.Path(
    profile_base_dir
    / "data"
    / "image_based_profiles_production_zedprofiler_pipeline_20260915_f5a6f16"
    / f"{patient}"
    # / f"{image_based_profiles_subparent_name}"
    / "3.annotated_profiles/nucleocentric_morphem_anno.parquet"
).resolve()
sammed_annotated_sc_profiles_path = pathlib.Path(
    profile_base_dir
    / "data"
    / "image_based_profiles_production_zedprofiler_pipeline_20260915_f5a6f16"
    / f"{patient}"
    # / f"{image_based_profiles_subparent_name}"
    / "3.annotated_profiles/sammed_sc_anno.parquet"
).resolve()


output_dir = pathlib.Path(
    profile_base_dir
    / "data"
    / "image_based_profiles_production_zedprofiler_pipeline_20260915_f5a6f16"
    / f"{patient}"
    # / f"{image_based_profiles_subparent_name}"
    / "4.qc_profiles"
)
output_dir.mkdir(parents=True, exist_ok=True)

sc_qc_output_path = pathlib.Path(f"{output_dir}/sc_flagged_outliers.parquet").resolve()
sammed_sc_qc_output_path = pathlib.Path(
    f"{output_dir}/sammed_sc_flagged_outliers.parquet"
).resolve()
nucleocentric_sammed_qc_output_path = pathlib.Path(
    f"{output_dir}/nucleocentric_sammed_flagged_outliers.parquet"
).resolve()
nucleocentric_morphem_qc_output_path = pathlib.Path(
    f"{output_dir}/nucleocentric_morphem_flagged_outliers.parquet"
).resolve()

orig_sc_profiles_df = pd.read_parquet(sc_file)
organoid_qc_profiles_df = pd.read_parquet(organoid_file)
# Print the shape and head of the combined organoid profiles DataFrame
print(orig_sc_profiles_df.shape)
orig_sc_profiles_df

(4434, 2659)


,Metadata_Biology_PatientID,Metadata_Experiment_PlateID,Metadata_Experiment_WellID,Metadata_Imaging_FieldID,Metadata_Imaging_ImageID,Metadata_Experiment_ImageSet,Metadata_Object_ObjectID,Metadata_Neighbors_NeighborsCountAdjacent,Metadata_Neighbors_NeighborsCountByDistance-10,Metadata_Experiment_WellFOV,...,Cytoplasm_ER-Mito_Colocalization_RankWeightedColocalizationCoeff1,Cytoplasm_ER-Mito_Colocalization_RankWeightedColocalizationCoeff2,Cytoplasm_AGP-Mito_Colocalization_Correlation,Cytoplasm_AGP-Mito_Colocalization_MandersCoeffM1,Cytoplasm_AGP-Mito_Colocalization_MandersCoeffM2,Cytoplasm_AGP-Mito_Colocalization_OverlapCoeff,Cytoplasm_AGP-Mito_Colocalization_MandersCoeffCostesM1,Cytoplasm_AGP-Mito_Colocalization_MandersCoeffCostesM2,Cytoplasm_AGP-Mito_Colocalization_RankWeightedColocalizationCoeff1,Cytoplasm_AGP-Mito_Colocalization_RankWeightedColocalizationCoeff2
0,SARCO361,SARCO361_T1,E11,3,SARCO361_T1__SARCO361_T1__E11__F3,SARCO361_T1__SARCO361_T1__E11__F3,1,0,0,E11-3,...,0.846983,0.851404,0.474307,0.559409,0.997567,3.370540,0.920064,0.916107,0.840752,0.811099
1,SARCO361,SARCO361_T1,E11,3,SARCO361_T1__SARCO361_T1__E11__F3,SARCO361_T1__SARCO361_T1__E11__F3,2,2,3,E11-3,...,0.783265,0.806018,0.466114,0.999971,0.470162,0.950662,1.000000,0.999467,0.762807,0.793605
2,SARCO361,SARCO361_T1,E11,3,SARCO361_T1__SARCO361_T1__E11__F3,SARCO361_T1__SARCO361_T1__E11__F3,3,2,2,E11-3,...,0.738757,0.760464,0.351009,1.000000,0.576389,0.937948,1.000000,0.987784,0.738977,0.765575
3,SARCO361,SARCO361_T1,E11,3,SARCO361_T1__SARCO361_T1__E11__F3,SARCO361_T1__SARCO361_T1__E11__F3,4,2,3,E11-3,...,0.777737,0.800018,0.493501,0.999476,0.483213,1.844231,0.978772,0.973622,0.797046,0.819608
4,SARCO361,SARCO361_T1,E11,3,SARCO361_T1__SARCO361_T1__E11__F3,SARCO361_T1__SARCO361_T1__E11__F3,5,2,2,E11-3,...,0.754251,0.770193,0.182392,0.999752,0.999934,1.148181,0.477968,0.375474,0.689771,0.683160
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4429,SARCO361,SARCO361_T1,E5,4,SARCO361_T1__SARCO361_T1__E5__F4,SARCO361_T1__SARCO361_T1__E5__F4,7,2,2,E5-4,...,0.714926,0.723925,0.129888,0.905003,0.999968,0.955405,0.971890,0.963129,0.689932,0.678444
4430,SARCO361,SARCO361_T1,E5,4,SARCO361_T1__SARCO361_T1__E5__F4,SARCO361_T1__SARCO361_T1__E5__F4,8,1,2,E5-4,...,0.740343,0.751503,0.102713,0.997017,0.999958,1.103869,0.978809,0.973632,0.705606,0.693701
4431,SARCO361,SARCO361_T1,E5,4,SARCO361_T1__SARCO361_T1__E5__F4,SARCO361_T1__SARCO361_T1__E5__F4,9,3,4,E5-4,...,0.809969,0.815062,0.396084,0.832257,0.966888,2.506614,0.981691,0.976310,0.795303,0.779469
4432,SARCO361,SARCO361_T1,E5,4,SARCO361_T1__SARCO361_T1__E5__F4,SARCO361_T1__SARCO361_T1__E5__F4,10,1,1,E5-4,...,0.828838,0.795943,0.385653,0.698875,0.635682,1.642524,0.999992,0.999977,0.865424,0.863183


In [5]:
sc_profiles_df = orig_sc_profiles_df.copy()
sc_profiles_df["Metadata_cqc_nan_detected"] = (
    sc_profiles_df[
        [
            "Metadata_Object_ObjectID",
            "Metadata_Object_ParentOrganoid",
            "Cell_NoChannel_VolumeSizeShape_Volume",
        ]
    ]
    .isna()
    .any(axis=1)
)
# Print the number of organoids flagged
flagged_count = sc_profiles_df["Metadata_cqc_nan_detected"].sum()
print(f"Number of organoids flagged: {flagged_count}")

sc_profiles_df.head()

Number of organoids flagged: 0


,Metadata_Biology_PatientID,Metadata_Experiment_PlateID,Metadata_Experiment_WellID,Metadata_Imaging_FieldID,Metadata_Imaging_ImageID,Metadata_Experiment_ImageSet,Metadata_Object_ObjectID,Metadata_Neighbors_NeighborsCountAdjacent,Metadata_Neighbors_NeighborsCountByDistance-10,Metadata_Experiment_WellFOV,...,Cytoplasm_ER-Mito_Colocalization_RankWeightedColocalizationCoeff2,Cytoplasm_AGP-Mito_Colocalization_Correlation,Cytoplasm_AGP-Mito_Colocalization_MandersCoeffM1,Cytoplasm_AGP-Mito_Colocalization_MandersCoeffM2,Cytoplasm_AGP-Mito_Colocalization_OverlapCoeff,Cytoplasm_AGP-Mito_Colocalization_MandersCoeffCostesM1,Cytoplasm_AGP-Mito_Colocalization_MandersCoeffCostesM2,Cytoplasm_AGP-Mito_Colocalization_RankWeightedColocalizationCoeff1,Cytoplasm_AGP-Mito_Colocalization_RankWeightedColocalizationCoeff2,Metadata_cqc_nan_detected
0,SARCO361,SARCO361_T1,E11,3,SARCO361_T1__SARCO361_T1__E11__F3,SARCO361_T1__SARCO361_T1__E11__F3,1,0,0,E11-3,...,0.851404,0.474307,0.559409,0.997567,3.370540,0.920064,0.916107,0.840752,0.811099,False
1,SARCO361,SARCO361_T1,E11,3,SARCO361_T1__SARCO361_T1__E11__F3,SARCO361_T1__SARCO361_T1__E11__F3,2,2,3,E11-3,...,0.806018,0.466114,0.999971,0.470162,0.950662,1.000000,0.999467,0.762807,0.793605,False
2,SARCO361,SARCO361_T1,E11,3,SARCO361_T1__SARCO361_T1__E11__F3,SARCO361_T1__SARCO361_T1__E11__F3,3,2,2,E11-3,...,0.760464,0.351009,1.000000,0.576389,0.937948,1.000000,0.987784,0.738977,0.765575,False
3,SARCO361,SARCO361_T1,E11,3,SARCO361_T1__SARCO361_T1__E11__F3,SARCO361_T1__SARCO361_T1__E11__F3,4,2,3,E11-3,...,0.800018,0.493501,0.999476,0.483213,1.844231,0.978772,0.973622,0.797046,0.819608,False
4,SARCO361,SARCO361_T1,E11,3,SARCO361_T1__SARCO361_T1__E11__F3,SARCO361_T1__SARCO361_T1__E11__F3,5,2,2,E11-3,...,0.770193,0.182392,0.999752,0.999934,1.148181,0.477968,0.375474,0.689771,0.683160,False


In [6]:
# Round 2: propagate organoid-level QC flags to single cells.
# A cell is flagged if its parent organoid was flagged in 7a.
# We match on (ParentOrganoid, WellFOV) rather than ParentOrganoid alone because
# object IDs are reassigned per-FOV and are not globally unique across the patient.

# Default QC flags
sc_profiles_df["Metadata_cqc_organoid_flagged"] = False
sc_profiles_df["Metadata_cqc_nan_detected"] = (
    sc_profiles_df[
        ["Metadata_Object_ObjectID", "Nuclei_NoChannel_VolumeSizeShape_Volume"]
    ]
    .isna()
    .any(axis=1)
)
sc_profiles_df["Metadata_cqc_missing_parent_organoid"] = (
    sc_profiles_df["Metadata_Object_ParentOrganoid"] == -1
)


organoid_flags_df = organoid_qc_profiles_df[
    ["Metadata_Object_ObjectID", "Metadata_Experiment_WellFOV"]
    + [col for col in organoid_qc_profiles_df.columns if col.startswith("Metadata_cqc")]
]

# Get flagged (object_id, image_set) pairs
flagged_pairs = set(
    organoid_flags_df.loc[
        organoid_flags_df.filter(like="cqc").any(axis=1),
        ["Metadata_Object_ObjectID", "Metadata_Experiment_WellFOV"],
    ].itertuples(index=False, name=None)
)

# Flag SC rows where both parent_organoid & image_set match a flagged organoid
sc_profiles_df["Metadata_cqc_organoid_flagged"] = sc_profiles_df.apply(
    lambda row: (
        (row["Metadata_Object_ParentOrganoid"], row["Metadata_Experiment_WellFOV"])
        in flagged_pairs
    ),
    axis=1,
)

print(sc_profiles_df.shape)
sc_profiles_df.head()

(4434, 2662)


,Metadata_Biology_PatientID,Metadata_Experiment_PlateID,Metadata_Experiment_WellID,Metadata_Imaging_FieldID,Metadata_Imaging_ImageID,Metadata_Experiment_ImageSet,Metadata_Object_ObjectID,Metadata_Neighbors_NeighborsCountAdjacent,Metadata_Neighbors_NeighborsCountByDistance-10,Metadata_Experiment_WellFOV,...,Cytoplasm_AGP-Mito_Colocalization_MandersCoeffM1,Cytoplasm_AGP-Mito_Colocalization_MandersCoeffM2,Cytoplasm_AGP-Mito_Colocalization_OverlapCoeff,Cytoplasm_AGP-Mito_Colocalization_MandersCoeffCostesM1,Cytoplasm_AGP-Mito_Colocalization_MandersCoeffCostesM2,Cytoplasm_AGP-Mito_Colocalization_RankWeightedColocalizationCoeff1,Cytoplasm_AGP-Mito_Colocalization_RankWeightedColocalizationCoeff2,Metadata_cqc_nan_detected,Metadata_cqc_organoid_flagged,Metadata_cqc_missing_parent_organoid
0,SARCO361,SARCO361_T1,E11,3,SARCO361_T1__SARCO361_T1__E11__F3,SARCO361_T1__SARCO361_T1__E11__F3,1,0,0,E11-3,...,0.559409,0.997567,3.370540,0.920064,0.916107,0.840752,0.811099,False,False,False
1,SARCO361,SARCO361_T1,E11,3,SARCO361_T1__SARCO361_T1__E11__F3,SARCO361_T1__SARCO361_T1__E11__F3,2,2,3,E11-3,...,0.999971,0.470162,0.950662,1.000000,0.999467,0.762807,0.793605,False,False,False
2,SARCO361,SARCO361_T1,E11,3,SARCO361_T1__SARCO361_T1__E11__F3,SARCO361_T1__SARCO361_T1__E11__F3,3,2,2,E11-3,...,1.000000,0.576389,0.937948,1.000000,0.987784,0.738977,0.765575,False,False,False
3,SARCO361,SARCO361_T1,E11,3,SARCO361_T1__SARCO361_T1__E11__F3,SARCO361_T1__SARCO361_T1__E11__F3,4,2,3,E11-3,...,0.999476,0.483213,1.844231,0.978772,0.973622,0.797046,0.819608,False,False,False
4,SARCO361,SARCO361_T1,E11,3,SARCO361_T1__SARCO361_T1__E11__F3,SARCO361_T1__SARCO361_T1__E11__F3,5,2,2,E11-3,...,0.999752,0.999934,1.148181,0.477968,0.375474,0.689771,0.683160,False,False,False


In [7]:
sc_profiles_df["Nuclei_NoChannel_VolumeSizeShape_Volume"].describe()

count    4434.000000
mean      389.316394
std       344.684745
min         5.120000
25%        97.180000
50%       329.085000
75%       580.995000
max      3132.260000
Name: Nuclei_NoChannel_VolumeSizeShape_Volume, dtype: float64

## Detect outlier single-cells using the non-flagged data

We will attempt to detect instances of poor quality segmentations using the nuclei compartment as the base. The conditions we are using are as follows:

1. Abnormally small or large nuclei using `Volume`
2. Abnormally high `mass displacement` in the nuclei for instances of mis-segmentation of background/no longer in-focus

In [8]:
# Set the metadata columns to be used in the QC process
metadata_columns = [x for x in sc_profiles_df.columns if "Metadata" in x]

In [9]:
# Round 3: nucleus-based outlier detection using z-score thresholds.
# Threshold sign: negative = flag below mean, positive = flag above mean.
# Threshold magnitude: number of standard deviations from the mean.
# Only cells that passed rounds 1 and 2 are evaluated here.
# Only process the rows that are not flagged
filtered_plate_df = sc_profiles_df[
    ~(
        sc_profiles_df["Metadata_cqc_nan_detected"]
        | sc_profiles_df["Metadata_cqc_organoid_flagged"]
        | sc_profiles_df["Metadata_cqc_missing_parent_organoid"]
    )
]

# --- Find size based nuclei outliers ---
print("Finding small nuclei outliers...")
small_nuclei_outliers = find_outliers(
    df=filtered_plate_df,
    metadata_columns=metadata_columns,
    feature_thresholds={
        "Nuclei_NoChannel_VolumeSizeShape_Volume": small_nuclei_threshold,  # Detect very small nuclei
    },
)

# Ensure the column exists before assignment
sc_profiles_df["Metadata_cqc_small_nuclei_outlier"] = False
sc_profiles_df.loc[small_nuclei_outliers.index, "Metadata_cqc_small_nuclei_outlier"] = (
    True
)

# Print number of outliers (only in filtered rows)
small_count = filtered_plate_df.index.intersection(small_nuclei_outliers.index).shape[0]
print(f"Small nuclei outliers found: {small_count}")

display(
    small_nuclei_outliers[
        [
            "Metadata_Experiment_PlateID",
            "Metadata_Experiment_WellFOV",
            "Nuclei_NoChannel_VolumeSizeShape_Volume",
            "Metadata_Object_ObjectID",
        ]
    ]
    .sort_values("Nuclei_NoChannel_VolumeSizeShape_Volume", ascending=False)
    .head()
)

print("Finding large nuclei outliers...")
large_nuclei_outliers = find_outliers(
    df=filtered_plate_df,
    metadata_columns=metadata_columns,
    feature_thresholds={
        "Nuclei_NoChannel_VolumeSizeShape_Volume": large_nuclei_threshold,  # Detect very large nuclei
    },
)

# Ensure the column exists before assignment
sc_profiles_df["Metadata_cqc_large_nuclei_outlier"] = False
sc_profiles_df.loc[large_nuclei_outliers.index, "Metadata_cqc_large_nuclei_outlier"] = (
    True
)

# Print number of outliers (only in filtered rows)
large_count = filtered_plate_df.index.intersection(large_nuclei_outliers.index).shape[0]
print(f"Large nuclei outliers found: {large_count}")

display(
    large_nuclei_outliers[
        [
            "Metadata_Experiment_PlateID",
            "Metadata_Experiment_WellFOV",
            "Nuclei_NoChannel_VolumeSizeShape_Volume",
            "Metadata_Object_ObjectID",
        ]
    ]
    .sort_values("Nuclei_NoChannel_VolumeSizeShape_Volume", ascending=True)
    .head()
)

# --- Find mass displacement based nuclei outliers ---
print("Finding high mass displacement outliers...")
high_mass_displacement_outliers = find_outliers(
    df=filtered_plate_df,
    metadata_columns=metadata_columns,
    feature_thresholds={
        "Nuclei_DNA_Intensity_MassDisplacement": high_mass_displacement_threshold,  # Detect high mass displacement
    },
)

# Ensure the column exists before assignment
sc_profiles_df["Metadata_cqc_mass_displacement_outlier"] = False
sc_profiles_df.loc[
    high_mass_displacement_outliers.index, "Metadata_cqc_mass_displacement_outlier"
] = True

# Print number of outliers (only in filtered rows)
high_mass_count = filtered_plate_df.index.intersection(
    high_mass_displacement_outliers.index
).shape[0]
print(f"High mass displacement outliers found: {high_mass_count}")

display(
    high_mass_displacement_outliers[
        [
            "Metadata_Experiment_PlateID",
            "Metadata_Experiment_WellFOV",
            "Nuclei_DNA_Intensity_MassDisplacement",
            "Metadata_Object_ObjectID",
        ]
    ]
    .sort_values("Nuclei_DNA_Intensity_MassDisplacement", ascending=True)
    .head()
)

# Save updated plate_df with flag columns included
sc_profiles_df.to_parquet(sc_qc_output_path, index=False)

Finding small nuclei outliers...
Number of outliers: 1412 (39.49%)
Outliers Range:
Nuclei_NoChannel_VolumeSizeShape_Volume Min: 5.120000000000001
Nuclei_NoChannel_VolumeSizeShape_Volume Max: 234.23000000000005
Small nuclei outliers found: 1412


,Metadata_Experiment_PlateID,Metadata_Experiment_WellFOV,Nuclei_NoChannel_VolumeSizeShape_Volume,Metadata_Object_ObjectID
1321,SARCO361_T1,D7-5,234.23,12
1375,SARCO361_T1,D3-4,234.03,7
203,SARCO361_T1,F7-7,233.60,7
2402,SARCO361_T1,D6-2,233.59,12
4403,SARCO361_T1,D8-5,233.59,10


Finding large nuclei outliers...
Number of outliers: 129 (3.61%)
Outliers Range:
Nuclei_NoChannel_VolumeSizeShape_Volume Min: 1083.4400000000003
Nuclei_NoChannel_VolumeSizeShape_Volume Max: 3132.2600000000007
Large nuclei outliers found: 129


,Metadata_Experiment_PlateID,Metadata_Experiment_WellFOV,Nuclei_NoChannel_VolumeSizeShape_Volume,Metadata_Object_ObjectID
316,SARCO361_T1,E2-7,1083.44,5
4327,SARCO361_T1,D3-5,1086.26,1
3632,SARCO361_T1,E9-5,1088.81,13
4340,SARCO361_T1,E9-3,1088.93,7
4336,SARCO361_T1,E9-3,1093.08,3


Finding high mass displacement outliers...
Number of outliers: 209 (5.84%)
Outliers Range:
Nuclei_DNA_Intensity_MassDisplacement Min: 0.9715881
Nuclei_DNA_Intensity_MassDisplacement Max: 2.7475657
High mass displacement outliers found: 209


,Metadata_Experiment_PlateID,Metadata_Experiment_WellFOV,Nuclei_DNA_Intensity_MassDisplacement,Metadata_Object_ObjectID
2039,SARCO361_T1,G11-4,0.971588,3
35,SARCO361_T1,F11-4,0.973213,10
833,SARCO361_T1,F10-4,0.976097,7
4426,SARCO361_T1,E5-4,0.979449,4
3247,SARCO361_T1,C10-5,0.981069,7


In [10]:
sc_profiles_df.head()

,Metadata_Biology_PatientID,Metadata_Experiment_PlateID,Metadata_Experiment_WellID,Metadata_Imaging_FieldID,Metadata_Imaging_ImageID,Metadata_Experiment_ImageSet,Metadata_Object_ObjectID,Metadata_Neighbors_NeighborsCountAdjacent,Metadata_Neighbors_NeighborsCountByDistance-10,Metadata_Experiment_WellFOV,...,Cytoplasm_AGP-Mito_Colocalization_MandersCoeffCostesM1,Cytoplasm_AGP-Mito_Colocalization_MandersCoeffCostesM2,Cytoplasm_AGP-Mito_Colocalization_RankWeightedColocalizationCoeff1,Cytoplasm_AGP-Mito_Colocalization_RankWeightedColocalizationCoeff2,Metadata_cqc_nan_detected,Metadata_cqc_organoid_flagged,Metadata_cqc_missing_parent_organoid,Metadata_cqc_small_nuclei_outlier,Metadata_cqc_large_nuclei_outlier,Metadata_cqc_mass_displacement_outlier
0,SARCO361,SARCO361_T1,E11,3,SARCO361_T1__SARCO361_T1__E11__F3,SARCO361_T1__SARCO361_T1__E11__F3,1,0,0,E11-3,...,0.920064,0.916107,0.840752,0.811099,False,False,False,False,False,True
1,SARCO361,SARCO361_T1,E11,3,SARCO361_T1__SARCO361_T1__E11__F3,SARCO361_T1__SARCO361_T1__E11__F3,2,2,3,E11-3,...,1.000000,0.999467,0.762807,0.793605,False,False,False,False,False,False
2,SARCO361,SARCO361_T1,E11,3,SARCO361_T1__SARCO361_T1__E11__F3,SARCO361_T1__SARCO361_T1__E11__F3,3,2,2,E11-3,...,1.000000,0.987784,0.738977,0.765575,False,False,False,True,False,False
3,SARCO361,SARCO361_T1,E11,3,SARCO361_T1__SARCO361_T1__E11__F3,SARCO361_T1__SARCO361_T1__E11__F3,4,2,3,E11-3,...,0.978772,0.973622,0.797046,0.819608,False,False,False,False,False,False
4,SARCO361,SARCO361_T1,E11,3,SARCO361_T1__SARCO361_T1__E11__F3,SARCO361_T1__SARCO361_T1__E11__F3,5,2,2,E11-3,...,0.477968,0.375474,0.689771,0.683160,False,False,False,True,False,False


### Merge the qc flags to the deep learning-based profiles and save the output
Merge the QC flags back to the original single cell profiles, which will be used in downstream analyses and single cell QC. 
We need to do this beacuase we do not run qc on black-box features. 
Merge on the Metadata_Biology_PatientTumor, Metadata_Experiment_WellFOV
and the Metadata_Object_ObjectID columns, which together uniquely identify each organoid profile row.

In [11]:
nucleocentric_annotated_sammed_df = pd.read_parquet(nucleocentric_annotated_sammed_path)
nucleocentric_annotated_morphem_df = pd.read_parquet(
    nucleocentric_annotated_morphem_output_path
)
sammed_annotated_sc_profiles_df = pd.read_parquet(sammed_annotated_sc_profiles_path)
df_dict = {
    "nulceocentric_sammed": {
        "df": nucleocentric_annotated_sammed_df,
        "qc_output_path": nucleocentric_sammed_qc_output_path,
    },
    "nucleocentric_chammi": {
        "df": nucleocentric_annotated_morphem_df,
        "qc_output_path": nucleocentric_morphem_qc_output_path,
    },
    "sammed_sc_profiles": {
        "df": sammed_annotated_sc_profiles_df,
        "qc_output_path": sammed_sc_qc_output_path,
    },
}

FileNotFoundError: [Errno 2] No such file or directory: '/home/jenna/mnt/bandicoot/NF1_organoid_data/data/image_based_profiles_production_zedprofiler_pipeline_20260915_f5a6f16/SARCO361_T1/3.annotated_profiles/nucleocentric_sammed_anno.parquet'

In [ ]:
# set the merge keys to int for both dataframes to ensure they match
merge_keys = [
    "Metadata_Biology_PatientTumor",
    "Metadata_Experiment_WellFOV",
    "Metadata_Object_ObjectID",
]
qc_keys = [col for col in sc_profiles_df.columns if "Metadata_cqc" in col]

for profile_name in df_dict:
    df = df_dict[profile_name]["df"]
    for key in merge_keys:
        if key not in df.columns:
            raise ValueError(f"Merge key {key} not found in dataframe columns.")
    qc_annotated_df = df.merge(
        sc_profiles_df[qc_keys + merge_keys],
        on=merge_keys,
        how="left",
    )
    if qc_annotated_df.shape[1] == df.shape[1]:
        raise ValueError(
            f"No new columns were added during the merge. Check that the merge keys {merge_keys} are correct and that the qc keys {qc_keys} are present in the sc_profiles_df."
        )
    qc_annotated_df.to_parquet(df_dict[profile_name]["qc_output_path"], index=False)